<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Exercise Solution: LLM Evaluation - Basic Metrics</h1>
Developed by WeCloudData
<br></br>




In [ ]:
!pip install evaluate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


## 1. BLEU Score

A BLEU score ranges between 0 and 1 and measures how similar the machine-translated text is to a set of high quality reference translations:
- A value of 0 means that the output has no overlap with the reference hence the quality is low.
- A value of 1 means that there is perfect overlap with the reference data. So how can we apply this metric to evaluate the quality of llm’s output.


In [ ]:
import evaluate
bleu = evaluate.load('bleu')
predictions = ["Transformers Transformers are fast plus efficient",
               "Good Morning", "I am waiting for new Transformers"]
references = [
              ["HuggingFace Transformers are quick, efficient and awesome",
               "Transformers are awesome because they are fast to execute"],
              ["Good Morning Transformers", "Morning Transformers"],
              ["People are eagerly waiting for new Transformer models",
               "People are very excited about new Transformers"]

]
results = bleu.compute(predictions=predictions,
                       references=references,
                       max_order = 2)
print(results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


{'bleu': 0.5037930378757725, 'precisions': [0.7142857142857143, 0.5454545454545454], 'brevity_penalty': 0.8071177470053892, 'length_ratio': 0.8235294117647058, 'translation_length': 14, 'reference_length': 17}


### BLEU Exercise

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)

def create_pipeline(
    task = 'summarization', model = 'facebook/bart-large-cnn', tokenizer = 'facebook/bart-large-cnn'
) -> pipeline:
    return pipeline(task=task, model=model, tokenizer=tokenizer)

summarizer = create_pipeline()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
!wget https://s3.amazonaws.com/weclouddata/datasets/llm/depression_information.txt

--2025-12-01 10:07:05--  https://s3.amazonaws.com/weclouddata/datasets/llm/depression_information.txt
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.216.50.152, 16.15.177.181, 54.231.132.120, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.216.50.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3440 (3.4K) [text/plain]
Saving to: ‘depression_information.txt’

depression_informat 100%[===================>]   3.36K  --.-KB/s    in 0s      

2025-12-01 10:07:05 (136 MB/s) - ‘depression_information.txt’ saved [3440/3440]



In [ ]:
def read_information(
    file_name = 'depression_information.txt'
) -> str:
    return open('depression_information.txt', 'r').read()

data = read_information()

In [ ]:
len(data)

3438

In [ ]:
data

'Overview - Depression in adults\nDepression is more than simply feeling unhappy or fed up for a few days.\n\nMost people go through periods of feeling down, but when you\'re depressed you feel persistently sad for weeks or months, rather than just a few days.\n\nSome people think depression is trivial and not a genuine health condition. They\'re wrong – it is a real illness with real symptoms. Depression is not a sign of weakness or something you can "snap out of" by "pulling yourself together".\n\nThe good news is that with the right treatment and support, most people with depression can make a full recovery.\n\nThese pages are about depression in adults. Read about depression in children and young people.\n\nHow to tell if you have depression\nDepression affects people in different ways and can cause a wide variety of symptoms.\n\nThey range from lasting feelings of unhappiness and hopelessness, to losing interest in the things you used to enjoy and feeling very tearful. Many people

In [ ]:
import evaluate

result = summarizer(data, max_length=500)
summarised_text = result[0]['summary_text']

bleu = evaluate.load('bleu')
scores = bleu.compute(predictions=[summarised_text], references=[data])

print(scores)

{'bleu': 0.00015729763123912478, 'precisions': [0.9696969696969697, 0.8923076923076924, 0.828125, 0.7619047619047619], 'brevity_penalty': 0.00018299337534830135, 'length_ratio': 0.10410094637223975, 'translation_length': 66, 'reference_length': 634}


In [ ]:
len(result)

1

In [ ]:
result

[{'summary_text': 'Depression affects people in different ways and can cause a wide variety of symptoms. Symptoms of depression range from mild to severe. At its mildest, you may simply feel persistently low in spirit. Severe depression can make you feel suicidal, that life is no longer worth living. Treatment for depression can involve a combination of lifestyle changes and medicine.'}]

In [ ]:
random_text = 'Thisa poor score should produce '
random_score = bleu.compute(predictions=[random_text], references=[data])
random_score

{'bleu': 0.0,
 'precisions': [0.3333333333333333, 0.0, 0.0, 0.0],
 'brevity_penalty': 3.4981967136826355e-46,
 'length_ratio': 0.00946372239747634,
 'translation_length': 6,
 'reference_length': 634}

## 2. Rouge Score


In [ ]:
!pip install rouge_score --quiet

  Preparing metadata (setup.py) ... done


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score('The quick brown dog jumps over the lazy fox.',
                      'The quick brown fox jumps over the lazy dog.')
print(scores)

{'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rougeL': Score(precision=0.7777777777777778, recall=0.7777777777777778, fmeasure=0.7777777777777778)}


### Rogue Score Exercise

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)

def create_pipeline(
    task = 'summarization', model = 'facebook/bart-large-cnn', tokenizer = 'facebook/bart-large-cnn'
) -> pipeline:
    return pipeline(task=task, model=model, tokenizer=tokenizer)

summarizer = create_pipeline()

Device set to use cuda:0


In [ ]:
def read_information(
    file_name = 'depression_information.txt'
) -> str:
    return open('depression_information.txt', 'r').read()

data = read_information()

In [ ]:
import evaluate

result = summarizer(data, max_length=500)
summarised_text = result[0]['summary_text']

rouge = evaluate.load('rouge')
scores = rouge.compute(predictions=[summarised_text], references=[data])

print(scores)

{'rouge1': np.float64(0.18849840255591052), 'rouge2': np.float64(0.17307692307692307), 'rougeL': np.float64(0.18849840255591052), 'rougeLsum': np.float64(0.18849840255591052)}


In [ ]:
summarised_text

'Depression affects people in different ways and can cause a wide variety of symptoms. Symptoms of depression range from mild to severe. At its mildest, you may simply feel persistently low in spirit. Severe depression can make you feel suicidal, that life is no longer worth living. Treatment for depression can involve a combination of lifestyle changes and medicine.'

## Meteor Score

In [ ]:
!pip install nltk

In [ ]:
from nltk.translate import meteor_score
from nltk import word_tokenize

import nltk

# Calculate the BLEU score
nltk.download('punkt_tab')
nltk.download('wordnet', download_dir='/usr/local/share/nltk_data')

reference = "The quick brown fox jumps over the lazy dog"
candidate = "The fast brown fox jumps over the lazy dog"
tokenized_reference = word_tokenize(reference)
tokenized_candidate = word_tokenize(candidate)
score = meteor_score.meteor_score([tokenized_reference], tokenized_candidate)
print(score)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /usr/local/share/nltk_data...


0.9993141289437586


- METEOR scores range from 0 to 1, with higher scores indicating better
similarity between the candidate and reference sentences.
- In this case, the high METEOR score of approximately 0.9993 indicates that the candidate sentence is very similar to the reference sentence according to the METEOR metric. This suggests that the candidate sentence is almost identical to the reference sentence in terms of lexical overlap, word order, and linguistic quality.

### Meteor Score Exercise

## BERTScore

In [ ]:
!pip install bert_score --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


In [ ]:
import torch
from bert_score import score


# reference and generated texts
ref_text = data
gen_text = summarised_text

# compute Bert score
P, R, F1 = score([gen_text], [ref_text], lang="en", model_type="bert-base-uncased")

# print results
print(f"Bert score: P={P.item():.4f} R={R.item():.4f} F1={F1.item():.4f}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Bert score: P=0.8812 R=0.5436 F1=0.6724


BERTScore is a metric for evaluating the quality of generated text, which leverages pre-trained BERT models.

- **Precision (P):** Precision measures the proportion of correctly identified relevant words in the generated summary compared to the reference text. A precision score of 0.8812 indicates that approximately 88.12% of the words in the generated summary are relevant to the reference text.

- **Recall (R):** Recall measures the proportion of relevant words in the reference text that are correctly identified in the generated summary. A recall score of 0.5436 indicates that approximately 54.36% of the relevant words in the reference text are present in the generated summary.